In [1]:
import torch
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

n_qubits = 2

x_params = ParameterVector("x", n_qubits)
w_params = ParameterVector("w", n_qubits)

circuit = QuantumCircuit(n_qubits)

for i in range(n_qubits):
    circuit.ry(x_params[i], i)
    circuit.rz(w_params[i], i)

circuit.cx(0, 1)

observables = [
    SparsePauliOp.from_list([("ZI", 1.0)]),
    SparsePauliOp.from_list([("IZ", 1.0)]),
]

qnn = EstimatorQNN(
    circuit=circuit,
    observables=observables,
    input_params=list(x_params),
    weight_params=list(w_params),
    input_gradients=True,
)

quantum_layer = TorchConnector(qnn)

x = torch.tensor([[0.2, -0.4]], dtype=torch.float32)
y = quantum_layer(x)

print("Output shape:", y.shape)
print("Output:", y)

Output shape: torch.Size([1, 2])
Output: tensor([[0.9027, 0.9801]], grad_fn=<_TorchNNFunctionBackward>)
